# 🎨 Topic 05: Channels, 3D Convolutions & Feature Maps
### 🧠 Multi-Channel Math, Custom 3D Conv from Scratch, Feature Map Visualizer & Parameter Calculations

---

## 📌 1. Multi-Channel Convolutions Kya Hoti Hain? (The 3D Picture)

Real-world computer vision mein images hamesha grayscale ($1$ channel) nahi hoti, balki color images hoti hain jisme **3 channels** hote hain:
- **Red (R)**
- **Green (G)**
- **Blue (B)**

### ❓ Question: Agar Input me 3 Channels hain, toh Filter kaisa hoga?
> **Answer**: Agar input tensor ki depth $C_{in}$ hai, toh **har filter ki depth bhi exactly $C_{in}$ honi chahiye!**
> - Input Shape: $(H, W, 3)$
> - Filter Shape: $(K_h, K_w, 3)$ *(Filter bhi 3D cube ban jata hai!)*

### ⚙️ 3D Convolution Kaise Kaam Karti Hai?
1. Filter ka Red slice image ke Red channel se convolve hota hai.
2. Filter ka Green slice image ke Green channel se convolve hota hai.
3. Filter ka Blue slice image ke Blue channel se convolve hota hai.
4. **Teeno channels ke outputs ko aapas mein ADD (sum) kiya jata hai**, aur aakhir mein **1 single bias ($b$) add hota hai**.
5. Result: **Sirf ek single 2D Feature Map ($H_{out} \times W_{out}$)** banta hai!

### ❓ Toh fir Output me multiple channels kaise aate hain?
> **Answer**: Agar hume output mein $16$ ya $32$ channels chahiye, toh hum **$16$ ya $32$ alag-alag 3D filters** use karte hain!
> - $1$ Filter $\rightarrow 1$ 2D Feature Map
> - $N$ Filters $\rightarrow N$ 2D Feature Maps stacked together $\rightarrow (H_{out}, W_{out}, N)$


In [ ]:
# ============================================================
# STEP 1: Libraries Import
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

print(f"TensorFlow Version: {tf.__version__}")


In [ ]:
# ============================================================
# STEP 2: Custom 3D Multi-Channel Convolution from Scratch (Pure NumPy)
# ============================================================

def conv3d_multi_filter_scratch(image_3d, filters_4d, biases):
    """
    image_3d: (H, W, C_in)
    filters_4d: (kH, kW, C_in, num_filters)
    biases: (num_filters,)
    Returns: (out_H, out_W, num_filters)
    """
    H, W, C_in = image_3d.shape
    kH, kW, f_C_in, num_filters = filters_4d.shape
    assert C_in == f_C_in, "Input channels aur filter channels match hone chahiye!"
    
    out_H = H - kH + 1
    out_W = W - kW + 1
    
    # Empty output tensor
    output_tensor = np.zeros((out_H, out_W, num_filters), dtype=np.float32)
    
    # Har filter ke liye loop
    for f in range(num_filters):
        current_filter = filters_4d[:, :, :, f]
        current_bias = biases[f]
        
        # Spatial sliding
        for i in range(out_H):
            for j in range(out_W):
                patch = image_3d[i:i + kH, j:j + kW, :]  # 3D patch
                # Teeno channels ka element-wise multiply aur sum across all 3 dimensions
                val = np.sum(patch * current_filter) + current_bias
                output_tensor[i, j, f] = val
                
    return output_tensor

# Testing with a dummy RGB image (4x4x3) and 2 filters of (3x3x3)
dummy_rgb = np.random.randn(4, 4, 3).astype(np.float32)
dummy_filters = np.random.randn(3, 3, 3, 2).astype(np.float32) # 2 filters
dummy_biases = np.array([0.5, -0.2], dtype=np.float32)

res_scratch_3d = conv3d_multi_filter_scratch(dummy_rgb, dummy_filters, dummy_biases)

print(f"Input RGB Image Shape:    {dummy_rgb.shape}")
print(f"Filters Shape (4D):       {dummy_filters.shape}")
print(f"Output Feature Maps Shape:{res_scratch_3d.shape} -> (2 Feature Maps of size 2x2)")


## 🧮 2. Masterclass: Parameter Calculation Formula

Har Deep Learning interview ka favorite question: **"Is layer mein kitne parameters hain?"**

$$\mathbf{\text{Conv2D Params}} = (K_h \times K_w \times C_{in} + 1) \times C_{out}$$

- $K_h \times K_w$ = Filter height and width (e.g. $3 \times 3 = 9$)
- $C_{in}$ = Input channels (previous layer ke feature maps)
- $+ 1$ = Bias term (har filter ka 1 bias)
- $C_{out}$ = Number of filters in current layer

### 📝 LeNet-5 Examples:
1. **Layer C1**: Input = $(32, 32, 1)$, Filters = $6$ of size $5 \times 5$
   $$\text{Params} = (5 \times 5 \times 1 + 1) \times 6 = 26 \times 6 = \mathbf{156}$$
2. **Layer C3**: Input = $(14, 14, 6)$, Filters = $16$ of size $5 \times 5$
   $$\text{Params} = (5 \times 5 \times 6 + 1) \times 16 = 151 \times 16 = \mathbf{2,416}$$
3. **Layer C5**: Input = $(5, 5, 16)$, Filters = $120$ of size $5 \times 5$
   $$\text{Params} = (5 \times 5 \times 16 + 1) \times 120 = 401 \times 120 = \mathbf{48,120}$$


## 🔬 3. Feature Map Visualizer (Network andar se kya dekh raha hai?)

Aaiye ek mini-CNN train karte hain aur uske **intermediate feature maps** ko extract karke visualise karte hain!
Dekhte hain ki Conv Layer 1 aur Conv Layer 2 real images par kya capture karte hain.

In [ ]:
# ============================================================
# STEP 3: Train Mini-CNN and Extract Feature Maps
# ============================================================

mnist = keras.datasets.mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = np.expand_dims(X_train.astype('float32') / 255.0, axis=-1)
X_test = np.expand_dims(X_test.astype('float32') / 255.0, axis=-1)

# Chota sa CNN model
input_layer = layers.Input(shape=(28, 28, 1), name="Input")
c1 = layers.Conv2D(8, kernel_size=(3, 3), activation='relu', padding='same', name="Conv_Layer_1")(input_layer)
p1 = layers.MaxPooling2D(pool_size=(2, 2), name="Pool_1")(c1)
c2 = layers.Conv2D(16, kernel_size=(3, 3), activation='relu', padding='same', name="Conv_Layer_2")(p1)
p2 = layers.MaxPooling2D(pool_size=(2, 2), name="Pool_2")(c2)
flat = layers.Flatten()(p2)
out = layers.Dense(10, activation='softmax', name="Output")(flat)

viz_model = models.Model(inputs=input_layer, outputs=out)
viz_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
viz_model.fit(X_train[:10000], y_train[:10000], epochs=2, batch_size=128, verbose=0)
print("✅ Mini-CNN trained successfully!")


In [ ]:
# ============================================================
# STEP 4: Feature Maps Visualisation
# ============================================================

# Intermediate Activation Model banate hain jo Conv1 aur Conv2 ke outputs dega
activation_model = models.Model(inputs=viz_model.input, outputs=[c1, c2])

# Sample test image lete hain
sample_img = X_test[0:1] # Digit 7
activations = activation_model.predict(sample_img)
conv1_maps, conv2_maps = activations

print(f"Sample Input Shape:     {sample_img.shape}")
print(f"Conv 1 Activation Shape:{conv1_maps.shape} -> (8 Feature Maps of size 28x28)")
print(f"Conv 2 Activation Shape:{conv2_maps.shape} -> (16 Feature Maps of size 14x14)")

# Plot Conv Layer 1 ke 8 Feature Maps
plt.figure(figsize=(14, 3))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(conv1_maps[0, :, :, i], cmap='viridis')
    plt.title(f"Filter {i+1}", fontsize=10, fontweight='bold')
    plt.axis('off')
plt.suptitle("Conv Layer 1: Low-level Edges & Boundaries (28x28)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Plot Conv Layer 2 ke 16 Feature Maps
plt.figure(figsize=(14, 4))
for i in range(16):
    plt.subplot(2, 8, i + 1)
    plt.imshow(conv2_maps[0, :, :, i], cmap='magma')
    plt.title(f"Filter {i+1}", fontsize=9)
    plt.axis('off')
plt.suptitle("Conv Layer 2: High-level Abstract Patterns & Corners (14x14)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 💡 Mind-blowing Observation:
- **Layer 1 Feature Maps (28x28)**: Inme edges, strokes, aur boundaries clearly recognizable hain (Low-level features).
- **Layer 2 Feature Maps (14x14)**: Yahan features aur abstract ban chuke hain (loops, corners, intersections).
- Jaise-jaise network deep hota jata hai, features simple lines se complex parts aur shapes mein convert hote hain!

## 🔗 The Road to LeNet-5

Ab aapne CNN ke saare core building blocks seekh liye hain:
1. **01: Why CNN over ANN** (Parameter explosion, spatial structure, translation invariance)
2. **02: Convolution & Filters** (Kernel sliding, matrix multiplication, edge detectors)
3. **03: Padding & Strides** (Valid vs Same padding, border preservation, downsampling)
4. **04: Pooling Layers** (Max, Average, Global pooling, spatial reduction)
5. **05: Channels & Feature Maps** (3D filters, multi-channel summation, parameter formula, intermediate representations)

👉 **Ab aap LeNet-5 (`lenet/lenet_5.ipynb`) ko bina kisi doubt ke 100% samajh sakte hain!**
